# 02 · Flood-specific EDA — official flood categories, events, lead time and feed delay
**Project:** Real-Time Flood & Flash-Flood Forecasting and Alerting using Apache Kafka (AI372IA, Phase I)

**Datasets:** USGS Instantaneous Values (river stage), NOAA NWPS gauge metadata (official *Action / Minor / Moderate / Major* flood stages), and hourly rainfall at the gauge from the Open-Meteo historical archive (a reanalysis used here only to explore rain → stage timing; the pipeline itself uses MRMS — see notebook 03).

**Questions this notebook answers for each candidate gauge**
1. How often does the river reach each official flood category?
2. How many separate flood events are there, and when do they happen?
3. How long does the river take to rise to flood stage (the ceiling on useful warning time)?
4. How stale is the live feed, compared with that rise time?
5. How long after rain does the river respond?
6. How well do simple baselines (persistence, rise-rate) already warn? — the bar our model has to beat.

**How to run:** Colab or Kaggle (Internet on). Fill `GAUGES` (USGS site → NWPS id; use `python -m tools.find_gauges` from the repo to find them), then *Run all*.

In [ ]:
# !pip install -q seaborn   # uncomment if needed

## 1. Setup and configuration

In [ ]:
import json, os, time, gzip, io, math
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import requests

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
UA = {"User-Agent": "rvce-flood-eda-notebook"}

def get_json(url, params=None, retries=3):
    """GET with retries; raises on final failure."""
    last = None
    for i in range(retries):
        try:
            r = requests.get(url, params=params, headers=UA, timeout=60)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last = e
            time.sleep(2 * (i + 1))
    raise RuntimeError(f"GET failed: {url} {params}: {last}")

def parse_iv(doc):
    """USGS Instantaneous Values JSON -> tidy DataFrame (one row per reading)."""
    rows, sites = [], {}
    for ts in doc["value"]["timeSeries"]:
        src, var = ts["sourceInfo"], ts["variable"]
        site = src["siteCode"][0]["value"]
        geo = src.get("geoLocation", {}).get("geogLocation", {})
        sites[site] = {"site_no": site, "site_name": src.get("siteName"),
                       "lat": geo.get("latitude"), "lon": geo.get("longitude")}
        param = var["variableCode"][0]["value"]
        nodata = var.get("noDataValue")
        for i, block in enumerate(ts.get("values", [])):
            meth = (block.get("method") or [{}])[0]
            series = str(meth.get("methodID", i))
            mdesc = meth.get("methodDescription", "")
            for pt in block.get("value", []):
                try:
                    v = float(pt["value"])
                except (TypeError, ValueError):
                    continue
                rows.append((site, param, series, mdesc, pt["dateTime"], v,
                             ",".join(pt.get("qualifiers") or []), v == nodata))
    df = pd.DataFrame(rows, columns=["site_no", "parameter", "series", "method", "dateTime",
                                     "value", "qualifiers", "is_nodata"])
    if len(df):
        df["datetime_utc"] = pd.to_datetime(df["dateTime"], utc=True, format="ISO8601")
        df = df.drop(columns="dateTime")
    return df, pd.DataFrame(sites.values())

def fetch_iv(sites, params, start, end, chunk_days=120, cache=True):
    """Fetch USGS IV data in chunks (the service limits request sizes)."""
    key = f"iv_{'-'.join(sites)}_{'-'.join(params)}_{start}_{end}.csv.gz"
    path = os.path.join(DATA_DIR, key)
    meta_path = path.replace(".csv.gz", "_sites.csv")
    if cache and os.path.exists(path) and os.path.exists(meta_path):
        df = pd.read_csv(path, dtype={"site_no": str, "parameter": str, "series": str})
        df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True, format="ISO8601")
        return df, pd.read_csv(meta_path, dtype={"site_no": str})
    frames, metas = [], []
    t0, t1 = pd.Timestamp(start), pd.Timestamp(end)
    cur = t0
    while cur < t1:
        nxt = min(cur + pd.Timedelta(days=chunk_days), t1)
        doc = get_json("https://waterservices.usgs.gov/nwis/iv/",
                       {"format": "json", "sites": ",".join(sites), "parameterCd": ",".join(params),
                        "startDT": cur.strftime("%Y-%m-%d"), "endDT": nxt.strftime("%Y-%m-%d"),
                        "siteStatus": "all"})
        d, m = parse_iv(doc)
        frames.append(d); metas.append(m)
        print(f"  {cur.date()} .. {nxt.date()}: {len(d):,} readings")
        cur = nxt + pd.Timedelta(days=1)
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    meta = pd.concat(metas).drop_duplicates("site_no") if metas else pd.DataFrame()
    df = df.drop_duplicates(["site_no", "parameter", "series", "datetime_utc"])
    if cache:
        df.to_csv(path, index=False); meta.to_csv(meta_path, index=False)
    return df, meta

def primary_series(df, site, param):
    """The series with the most readings is treated as the primary sensor."""
    d = df[(df.site_no == site) & (df.parameter == param) & (~df.is_nodata)]
    if d.empty:
        return pd.Series(dtype=float)
    s = d.series.value_counts().idxmax()
    out = d[d.series == s].set_index("datetime_utc")["value"].sort_index()
    return out[~out.index.duplicated()]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

# ---- CONFIG ----
GAUGES = {                       # USGS site number -> NWPS gauge id (lid)
    "08158700": "DRWT2",         # Onion Creek near Driftwood: slow, 16 events, passes the gate (5-min data)
    "08172400": "LCPT2",         # Plum Creek at Lockhart: slow, 15 events, passes the gate (15-min data)
    "08150000": "JNCT2",         # Llano River near Junction: slow, 9 events (pooled set only)
    "08156800": "AHOT2",         # Shoal Creek, Austin: fast urban creek, 38 events, median rise < 1 h (contrast)
}
MANUAL_THRESHOLDS = {            # used only when a gauge has no NWPS lid (ft): action, minor, moderate, major
    # "08158700": {"action": None, "minor": None, "moderate": None, "major": None},
}
YEARS      = 10                  # history to analyse
CATEGORY   = "minor"             # flood category used for event detection
MIN_GAP_H  = 24                  # readings above threshold closer than this = same event
LOOKBACK_H = 48                  # how far back to search for the start of a rise
MIN_EVENTS, MIN_RISE_H = 10, 3.0 # Phase-0 gate rules
END   = pd.Timestamp.now(tz="UTC").normalize().tz_localize(None)
START = END - pd.Timedelta(days=365 * YEARS)
CATS  = ["action", "minor", "moderate", "major"]

## 2. Official flood categories (NWPS)

In [ ]:
def nwps_categories(lid):
    doc = get_json(f"https://api.water.noaa.gov/nwps/v1/gauges/{lid}")
    cats = (doc.get("flood") or {}).get("categories") or {}
    out = {}
    for c in CATS:
        v = (cats.get(c) or {}).get("stage")
        out[c] = None if v is None or float(v) == -9999 else float(v)
    out.update(lid=lid, name=doc.get("name"), usgs_id=doc.get("usgsId"),
               units=(doc.get("flood") or {}).get("stageUnits"))
    return out

thr = {}
for site, lid in GAUGES.items():
    if lid:
        thr[site] = nwps_categories(lid)
    elif site in MANUAL_THRESHOLDS:
        thr[site] = {**MANUAL_THRESHOLDS[site], "lid": None, "name": "(manual)", "usgs_id": site, "units": "ft"}
    else:
        print(f"{site}: no NWPS id and no manual thresholds -> category analysis skipped")
thr_df = pd.DataFrame(thr).T
thr_df

## 3. Stage history

In [ ]:
SITES = list(GAUGES)
df, sites_meta = fetch_iv(SITES, ["00065"], START.date().isoformat(), END.date().isoformat(), chunk_days=180)
stage = {s: primary_series(df, s, "00065") for s in SITES}
pd.DataFrame({s: {"readings": len(v), "from": v.index.min(), "to": v.index.max()} for s, v in stage.items()}).T

## 4. How often is each flood category reached?

In [ ]:
rows = []
for s, v in stage.items():
    if s not in thr or v.empty: continue
    for c in CATS:
        t = thr[s].get(c)
        if t is None: continue
        rows.append({"site_no": s, "category": c, "stage_ft": t,
                     "% of readings at/above": round((v >= t).mean() * 100, 3),
                     "days with any reading at/above": int((v >= t).resample("D").max().sum())})
pd.DataFrame(rows)

In [ ]:
# Stage-duration curve: % of time the stage is exceeded, with the official categories marked
for s, v in stage.items():
    if v.empty: continue
    srt = np.sort(v.values)[::-1]; exc = np.arange(1, len(srt) + 1) / len(srt) * 100
    plt.figure(figsize=(9, 4)); plt.plot(exc, srt); plt.xscale("log")
    for c, col in zip(CATS, ["gold", "orange", "red", "purple"]):
        t = thr.get(s, {}).get(c)
        if t is not None: plt.axhline(t, color=col, ls="--", label=f"{c} {t} ft")
    plt.xlabel("% of time exceeded (log)"); plt.ylabel("stage (ft)"); plt.title(f"{s}: stage-duration curve")
    plt.legend(); plt.show()

## 5. Flood events
An **event** starts at the first reading at/above the category after the river was below it; readings above it within `MIN_GAP_H` hours belong to the same event. The **rise start** is the nearest trough before the crossing; **rise time = crossing − rise start**. These are the same definitions the pipeline's gate check uses (`common/hydro.py`).

In [ ]:
def detect_events(v, threshold, min_gap_h=MIN_GAP_H, lookback_h=LOOKBACK_H, tol=0.1):
    t = v.index; x = v.values; ev = []; last_above = None; prev_below = True
    for i in range(len(x)):
        if x[i] >= threshold:
            new = last_above is None or (t[i] - last_above) > pd.Timedelta(hours=min_gap_h)
            if prev_below and new:
                ev.append(i)
            last_above = t[i]; prev_below = False
        else:
            prev_below = True
    rows = []
    for i in ev:
        j = best = i                                   # nearest trough before the crossing
        while j > 0 and t[i] - t[j - 1] <= pd.Timedelta(hours=lookback_h):
            j -= 1
            if x[j] < x[best]: best = j
            elif x[j] > x[best] + tol: break
        win = v[t[i]: t[i] + pd.Timedelta(hours=72)]
        above = v[t[i]:][v[t[i]:] < threshold]
        end = above.index[0] if len(above) else t[-1]
        rows.append({"crossing": t[i], "rise_start": t[best], "rise_start_ft": x[best],
                     "rise_time_h": (t[i] - t[best]).total_seconds() / 3600,
                     "peak_time": win.idxmax(), "peak_ft": win.max(),
                     "time_to_peak_h": (win.idxmax() - t[i]).total_seconds() / 3600,
                     "hours_above": (end - t[i]).total_seconds() / 3600})
    return pd.DataFrame(rows)

events = {}
for s, v in stage.items():
    t = thr.get(s, {}).get(CATEGORY)
    if t is None or v.empty: continue
    events[s] = detect_events(v, t)
    print(f"{s}: {len(events[s])} events at/above {CATEGORY} ({t} ft)")
all_events = pd.concat([e.assign(site_no=s) for s, e in events.items()], ignore_index=True) if events else pd.DataFrame()
all_events.head(20)

In [ ]:
if len(all_events):
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
    all_events.groupby([all_events.crossing.dt.year, "site_no"]).size().unstack(fill_value=0).plot.bar(ax=ax[0])
    ax[0].set_title("events per year"); ax[0].set_xlabel("")
    all_events.groupby([all_events.crossing.dt.month, "site_no"]).size().unstack(fill_value=0).reindex(range(1, 13), fill_value=0).plot.bar(ax=ax[1])
    ax[1].set_title("events by month (seasonality)"); ax[1].set_xlabel("month")
    sns.histplot(data=all_events, x="peak_ft", hue="site_no", ax=ax[2]); ax[2].set_title("event peak stage")
    plt.tight_layout(); plt.show()

## 6. Rise time — how much warning does the river itself give?

In [ ]:
if len(all_events):
    sns.histplot(data=all_events, x="rise_time_h", hue="site_no", bins=30)
    plt.axvline(MIN_RISE_H, color="red", ls="--", label=f"gate rule {MIN_RISE_H} h"); plt.legend()
    plt.xlabel("rise time to flood stage (h)"); plt.title("Rise time distribution"); plt.show()
    display(all_events.groupby("site_no")[["rise_time_h", "time_to_peak_h", "hours_above"]].describe().T.round(2))

In [ ]:
# Event hydrographs aligned at the crossing (t = 0)
for s, e in events.items():
    if e.empty: continue
    plt.figure(figsize=(10, 4))
    for _, r in e.iterrows():
        w = stage[s][r.crossing - pd.Timedelta(hours=24): r.crossing + pd.Timedelta(hours=24)]
        plt.plot((w.index - r.crossing).total_seconds() / 3600, w.values, alpha=.5)
    plt.axvline(0, color="k", lw=1); plt.axhline(thr[s][CATEGORY], color="orange", ls="--", label=CATEGORY)
    plt.xlabel("hours from crossing"); plt.ylabel("stage (ft)"); plt.title(f"{s}: all events aligned at the crossing")
    plt.legend(); plt.show()

## 7. Rise rates — normal behaviour vs. the hours before a flood

In [ ]:
for s, v in stage.items():
    if v.empty: continue
    h = v.resample("h").mean().interpolate(limit=3)
    rr = h.diff()                                       # ft per hour
    pre = pd.Series(False, index=rr.index)
    for c in events.get(s, pd.DataFrame()).get("crossing", []):
        pre[(rr.index > c - pd.Timedelta(hours=6)) & (rr.index <= c)] = True
    d = pd.DataFrame({"rise_ft_per_h": rr, "6 h before a crossing": pre}).dropna()
    sns.boxplot(data=d, x="6 h before a crossing", y="rise_ft_per_h", showfliers=False)
    plt.title(f"{s}: hourly rise rate, normal hours vs the 6 h before a flood"); plt.show()
    display(d.groupby("6 h before a crossing")["rise_ft_per_h"].describe().round(3))

## 8. Live feed delay (how old is the newest reading when we fetch it?)

In [ ]:
now_doc = get_json("https://waterservices.usgs.gov/nwis/iv/",
                   {"format": "json", "sites": ",".join(SITES), "parameterCd": "00065", "period": "PT6H"})
live, _ = parse_iv(now_doc)
fetched = pd.Timestamp.now(tz="UTC")
lat = (live.groupby(["site_no", "series"])
           .agg(newest=("datetime_utc", "max"), readings_6h=("value", "size"),
                provisional_share=("qualifiers", lambda q: q.str.contains("P").mean()))
           .reset_index())
lat["delay_min"] = ((fetched - lat.newest).dt.total_seconds() / 60).round(1)
lat

One fetch is a single sample. The repo's `tools/staleness_logger.py` records this every 5 minutes for days; its CSV can be loaded here for the full distribution. **Rule of thumb for the gate:** median rise time must comfortably exceed the typical delay, otherwise the alert arrives after the river has already risen.

In [ ]:
if len(all_events):
    comp = all_events.groupby("site_no")["rise_time_h"].median().rename("median_rise_h").to_frame()
    comp["live_delay_h"] = lat.groupby("site_no")["delay_min"].min() / 60
    comp["usable_warning_h"] = comp.median_rise_h - comp.live_delay_h
    display(comp.round(2))

## 9. Rain → stage response (Open-Meteo hourly rainfall at the gauge)

In [ ]:
def hourly_rain(lat_, lon_, start, end):
    doc = get_json("https://archive-api.open-meteo.com/v1/archive",
                   {"latitude": lat_, "longitude": lon_, "start_date": start, "end_date": end,
                    "hourly": "precipitation", "timezone": "UTC"})
    h = doc["hourly"]
    return pd.Series(h["precipitation"], index=pd.to_datetime(h["time"], utc=True), name="rain_mm")

lags = []
for s, e in events.items():
    if e.empty: continue
    m = sites_meta.set_index("site_no").loc[s]
    for _, r in e.sort_values("peak_ft", ascending=False).head(5).iterrows():
        t0, t1 = r.crossing - pd.Timedelta(days=3), r.crossing + pd.Timedelta(days=2)
        try:
            rain = hourly_rain(m.lat, m.lon, t0.date().isoformat(), t1.date().isoformat())
        except Exception as ex:
            print("rain fetch failed:", ex); continue
        st = stage[s][t0:t1].resample("h").mean().interpolate(limit=3)
        j = pd.concat([rain, st.rename("stage")], axis=1).dropna()
        if len(j) < 24 or j.rain_mm.sum() == 0: continue
        cc = {L: j.rain_mm.corr(j.stage.diff().shift(-L)) for L in range(0, 25)}
        best = max(cc, key=lambda L: -1 if pd.isna(cc[L]) else cc[L])
        lags.append({"site_no": s, "crossing": r.crossing, "rain_mm_total": j.rain_mm.sum(),
                     "best_lag_h": best, "corr_at_lag": round(cc[best], 2)})
        fig, ax1 = plt.subplots(figsize=(11, 3.5)); ax2 = ax1.twinx()
        ax1.bar(j.index, j.rain_mm, width=1 / 24, color="tab:blue", alpha=.5); ax1.set_ylabel("rain (mm/h)")
        ax2.plot(j.index, j.stage, color="tab:red"); ax2.set_ylabel("stage (ft)")
        ax2.axhline(thr[s][CATEGORY], color="orange", ls="--")
        plt.title(f"{s}: rain vs stage, event crossing {r.crossing:%Y-%m-%d %H:%M} UTC (best lag {best} h)"); plt.show()
pd.DataFrame(lags)

## 10. How well do simple baselines already warn?

In [ ]:
def baseline_scores(v, threshold, horizon_h, model, cooldown_h=6):
    h = v.resample("h").mean().interpolate(limit=3).dropna()
    rate = h.diff()
    fc = h if model == "persistence" else h + rate.clip(lower=0) * horizon_h
    alerts, last = [], None
    for t in h.index:
        if fc[t] >= threshold > h[t] and (last is None or t - last >= pd.Timedelta(hours=cooldown_h)):
            alerts.append(t); last = t
    ev = detect_events(v, threshold)
    crossings = list(ev.crossing) if len(ev) else []
    H = pd.Timedelta(hours=horizon_h)
    tp = sum(any(a < c <= a + H for c in crossings) for a in alerts)
    leads = []
    for c in crossings:
        prior = [a for a in alerts if c - H <= a < c]
        if prior: leads.append((c - min(prior)).total_seconds() / 3600)
    p = tp / len(alerts) if alerts else np.nan
    r = len(leads) / len(crossings) if crossings else np.nan
    f1 = 2 * p * r / (p + r) if p and r and (p + r) > 0 else np.nan
    err = (fc.shift(freq=pd.Timedelta(hours=horizon_h)) - h).dropna()
    return {"model": model, "horizon_h": horizon_h, "alerts": len(alerts), "false_alarms": len(alerts) - tp,
            "events": len(crossings), "detected": len(leads), "precision": p, "recall": r, "f1": f1,
            "median_lead_h": np.median(leads) if leads else np.nan, "MAE_ft": err.abs().mean()}

res = []
for s, v in stage.items():
    t = thr.get(s, {}).get(CATEGORY)
    if t is None or v.empty: continue
    for H in (1, 2, 3, 6):
        for mdl in ("persistence", "rise_rate"):
            res.append({"site_no": s, **baseline_scores(v, t, H, mdl)})
baselines = pd.DataFrame(res).round(3)
baselines

Persistence can never warn *before* a crossing (it predicts no change), so its recall is 0 by construction; it is still the right yardstick for stage error (MAE). Rise-rate extrapolation is the simplest model that can warn early — the ML forecaster must beat both.

## 11. Gate summary for each gauge

In [ ]:
rows = []
for s in SITES:
    e = events.get(s, pd.DataFrame())
    n = len(e); med = e.rise_time_h.median() if n else np.nan
    rows.append({"site_no": s, "lid": GAUGES[s], "events": n, "median_rise_h": med,
                 "verdict": "PASS" if n >= MIN_EVENTS and med >= MIN_RISE_H else "FAIL",
                 "reason": "" if n >= MIN_EVENTS and med >= MIN_RISE_H else
                           (f"only {n} events" if n < MIN_EVENTS else f"median rise {med:.1f} h < {MIN_RISE_H} h")})
gate = pd.DataFrame(rows); gate.to_csv(os.path.join(DATA_DIR, "eda02_gate_summary.csv"), index=False)
if len(all_events): all_events.to_csv(os.path.join(DATA_DIR, "eda02_events.csv"), index=False)
gate